##**Quantitative Translational Imaging in Medicine Lab — Summer Scholar Program**

<div style="background-color:black; padding:20px; text-align:center; border-radius: 8px;">
    <img src="https://i0.wp.com/www.martinos.org/wp-content/uploads/2019/01/spark_no_fade.gif?fit=404%2C303&ssl=1" alt="Martinos Center Logo" width="400"/>
    <h1 style="color:white; font-family: sans-serif;">Martinos Center for Biomedical Imaging</h1>
</div>



# Day 5: Building, Training, and Understanding Your AI Model

**Welcome back!** Yesterday (Day 4) you loaded, explored, and preprocessed two medical
imaging datasets — ChestMNIST and PathMNIST — and built a data augmentation pipeline.

Today you'll actually build a neural network, train it, evaluate how well it works, and look
inside it to see what it's "paying attention to."

### Quick recap & setup

If you're starting a brand new Colab session today, none of yesterday's variables are still
in memory — Colab doesn't remember anything between sessions. The cells below rebuild the key
pieces from yesterday (the same values you calculated yourself) so we can jump back in. This
is recap, not new material — feel free to skim the comments rather than re-deriving anything.

In [ ]:
# Same installs as yesterday - run this first
!pip install torch torchvision torchaudio
!pip install medmnist
!pip install matplotlib numpy scikit-learn

In [ ]:
# Same imports as yesterday
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from tqdm import tqdm

import medmnist
from medmnist import INFO, Evaluator

import matplotlib.pyplot as plt
import numpy as np

print(f"MedMNIST v{medmnist.__version__} is installed.")

### Rebuilding ChestMNIST setup

This recreates the normalization values and transform pipelines you calculated yourself in
Module 3 yesterday.

In [ ]:
# Recalculate ChestMNIST mean/std (same as yesterday's Module 3)
temp_dataset = medmnist.ChestMNIST(split='train', transform=transforms.ToTensor(), download=True)
temp_loader = DataLoader(dataset=temp_dataset, batch_size=1024, shuffle=False)

psum = torch.tensor([0.0])
psum_sq = torch.tensor([0.0])
for inputs, _ in tqdm(temp_loader, desc="Calculating ChestMNIST Mean/Std"):
    psum += inputs.sum(axis=[0, 2, 3])
    psum_sq += (inputs**2).sum(axis=[0, 2, 3])

count = len(temp_dataset) * 28 * 28
chest_mean = psum / count
chest_std = torch.sqrt((psum_sq / count) - (chest_mean ** 2))

chest_augmentation_transform = transforms.Compose([
    transforms.RandomRotation(5),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ToTensor(),
    transforms.Normalize(mean=chest_mean, std=chest_std)
])

chest_data_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=chest_mean, std=chest_std)
])

print(f"ChestMNIST mean: {chest_mean.tolist()}, std: {chest_std.tolist()}")
print("ChestMNIST transforms ready.")

### Rebuilding PathMNIST setup

Same idea, but for PathMNIST (which has 3 color channels instead of 1, so the mean/std are
each a list of 3 numbers).

In [ ]:
# Recreate the PathMNIST label info dictionary (from Module 1 yesterday)
path_info = INFO['pathmnist']
path_label_map = path_info['label']

# Recalculate PathMNIST mean/std (same as yesterday's Module 3 "Your Turn")
temp_path_dataset = medmnist.PathMNIST(split='train', transform=transforms.ToTensor(), download=True)
temp_path_loader = DataLoader(dataset=temp_path_dataset, batch_size=1024, shuffle=False)

psum = torch.tensor([0.0, 0.0, 0.0])
psum_sq = torch.tensor([0.0, 0.0, 0.0])
for inputs, _ in tqdm(temp_path_loader, desc="Calculating PathMNIST Mean/Std"):
    psum += inputs.sum(axis=[0, 2, 3])
    psum_sq += (inputs**2).sum(axis=[0, 2, 3])

count = len(temp_path_dataset) * 28 * 28
path_mean = psum / count
path_std = torch.sqrt((psum_sq / count) - (path_mean ** 2))

path_train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=path_mean, std=path_std)
])

path_val_test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=path_mean, std=path_std)
])

print(f"PathMNIST mean: {path_mean.tolist()}, std: {path_std.tolist()}")
print("PathMNIST transforms ready.")

Setup complete! Everything below picks up exactly where Day 4 left off.

---

---
## Module 4: Building Your First Neural Network

Welcome to Day 2! It's time to build our model. We will use a **Convolutional Neural Network (CNN)**, which is the standard type of AI for understanding images.

Think of a CNN as a series of filters and processors. Early layers might learn to see simple things like edges and corners. Deeper layers combine that knowledge to see more complex patterns like textures, shapes, and eventually, the features of a disease.

A CNN consists of several key layers:
- **`Conv2d` (Convolutional Layer)**: The workhorse. It uses small filters (like a tiny magnifying glass) to scan the image and detect features.
- **`ReLU` (Activation Function)**: A simple but powerful layer that introduces non-linearity. It's like an on/off switch for the features detected, allowing the model to learn complex relationships.
- **`MaxPool2d` (Pooling Layer)**: This layer shrinks the image, keeping only the most important information. This makes the model faster and helps it focus on the bigger picture.
- **`Linear` (Fully-Connected Layer)**: After the features have been extracted, these layers act like a traditional classifier to make the final decision based on the evidence found.

### Part A (Example): A Simple CNN for `ChestMNIST`

We will define a CNN class in PyTorch. A Python `class` is like a blueprint for creating something. Our `SimpleCNN` blueprint will tell PyTorch exactly what layers to create and how to connect them.

Pay attention to:
- `in_channels=1`: The first layer must accept 1-channel (grayscale) images.
- `num_classes=14`: The final layer must have 14 outputs, one for each disease class.
- **The `forward` method**: This defines the path the data takes through the layers.

In [ ]:
class SimpleCNN(nn.Module):
    # This is the blueprint for our model
    def __init__(self, in_channels, num_classes):
        super(SimpleCNN, self).__init__()

        # Here we define the layers of our network
        self.conv_block_1 = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=3, padding=1), # Input: 1x28x28, Output: 16x28x28
            nn.BatchNorm2d(16), # Normalizes the output of the conv layer
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2) # Output: 16x14x14
        )

        self.conv_block_2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, padding=1), # Input: 16x14x14, Output: 32x14x14
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2) # Output: 32x7x7
        )

        # The final classifier layer. It takes the flattened output of the conv blocks.
        # The size is 32 (channels) * 7 (height) * 7 (width)
        self.classifier = nn.Linear(32 * 7 * 7, num_classes)

    # The forward method defines the path data takes through the network
    def forward(self, x):
        x = self.conv_block_1(x)
        x = self.conv_block_2(x)
        # Flatten the output from a 2D feature map to a 1D vector
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

# Create an instance of our model for ChestMNIST
chest_model = SimpleCNN(in_channels=1, num_classes=14)
print(chest_model)

### Part B (Project): Adapting the CNN for `PathMNIST`

Now it's your turn. You need to create an instance of the `SimpleCNN` for the `PathMNIST` project. You can re-use the exact same blueprint, but you need to tell it the correct `in_channels` and `num_classes` for our project.

#### Your Turn: Instantiate the Model for PathMNIST

Create an instance of the `SimpleCNN` class for the `PathMNIST` task. Answer the questions in the comments to figure out the right parameters.

In [ ]:
# Question 1: PathMNIST images are RGB. How many input channels does that mean?
input_channels_for_path = # --- YOUR ANSWER HERE ---

# Question 2: How many different tissue types (classes) are in PathMNIST?
num_classes_for_path = # --- YOUR ANSWER HERE ---

# Task: Create an instance of the SimpleCNN for the PathMNIST task using your answers above.
path_model = # --- YOUR CODE HERE ---

print("PathMNIST Model Blueprint:")
print(path_model)

---
## Module 5: Teaching the AI and Checking Its Work

This is where the magic happens. We're going to teach our models by showing them the data. This process is called **training**.

### Key Components of Training:
1.  **Loss Function**: This is the "teacher". It looks at the model's prediction and compares it to the true label. It then calculates a "score" of how wrong the model was. The model's goal is to get this score as low as possible.
    - We'll use `nn.BCEWithLogitsLoss` for `ChestMNIST` (because it's multi-label).
    - We'll use `nn.CrossEntropyLoss` for `PathMNIST` (the standard for multi-class).
2.  **Optimizer**: This is the "student". It takes the score from the loss function and adjusts the model's internal parameters (weights) slightly to try and do better next time. We'll use `Adam`, a popular and effective optimizer.
3.  **Epochs**: One epoch is one full pass through the entire training dataset. We usually train for multiple epochs.

### Part A (Example): Training the `ChestMNIST` model

Here is the full training process for the `ChestMNIST` model. We will break it down into three parts:
1.  **Setup**: Preparing the datasets, dataloaders, model, and training components.
2.  **The Training Loop**: The code that actually teaches the model.
3.  **Evaluation**: Checking how well the model learned.

#### Step 1: Setup
First, we'll set up everything we need for training.

In [ ]:
from sklearn.utils import shuffle
# Set the device to use a GPU if available, otherwise use the CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Reload datasets with the proper augmentation/normalization transform
chest_train_dataset = medmnist.ChestMNIST(split='train', transform=chest_augmentation_transform, download=True)
chest_val_dataset = medmnist.ChestMNIST(split='val', transform=chest_data_transform, download=True)
chest_test_dataset = medmnist.ChestMNIST(split='test', transform=chest_data_transform, download=True)

# Create DataLoaders
chest_train_loader = DataLoader(dataset=chest_train_dataset, batch_size=128, shuffle=True)
chest_val_loader = DataLoader(dataset=chest_val_dataset, batch_size=128, shuffle=False)
chest_test_loader = DataLoader(dataset=chest_test_dataset, batch_size=128, shuffle=True)

# --- Training components for ChestMNIST ---
model = SimpleCNN(in_channels=1, num_classes=14).to(device)
criterion = nn.BCEWithLogitsLoss() # The 'teacher'
optimizer = optim.Adam(model.parameters(), lr=0.001) # The 'student'

NUM_EPOCHS = 3

#### Step 2: The Training Loop
This is the core of the process. For each epoch, we loop through all the batches in our training data. For each batch, we perform the forward pass, calculate loss, and then the backward pass to update the model. While you wait for the training to finish, you can check out some more details about CNN training in this [video](https://www.youtube.com/watch?v=QzY57FaENXg).

In [ ]:
# --- The Training Loop ---
for epoch in range(NUM_EPOCHS):
    model.train() # Set the model to training mode
    # The tqdm library gives us a nice progress bar
    for images, labels in tqdm(chest_train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} Training"):
        # 1. Move data to the GPU/CPU and ensure labels are float
        images, labels = images.to(device), labels.float().to(device)

        # 2. Forward pass: get the model's predictions
        outputs = model(images)

        # 3. Calculate the loss (how wrong the model was)
        loss = criterion(outputs, labels)

        # 4. Backward pass and optimize
        optimizer.zero_grad() # Reset the gradients
        loss.backward()       # Calculate the gradients
        optimizer.step()      # Update the model's weights

    # --- Validation Phase ---
    model.eval() # Set the model to evaluation mode (disables things like dropout)
    y_true = torch.tensor([]).to(device)
    y_score = torch.tensor([]).to(device)
    with torch.no_grad(): # We don't need to calculate gradients for validation
        for images, labels in chest_val_loader:
            images, labels = images.to(device), labels.float().to(device)
            outputs = model(images)
            y_true = torch.cat((y_true, labels), 0)
            y_score = torch.cat((y_score, outputs), 0)

    # Use the MedMNIST evaluator to calculate metrics like Area Under Curve (AUC)
    y_true = y_true.cpu().numpy()
    y_score = torch.sigmoid(y_score).cpu().numpy() # Apply sigmoid to convert raw scores to probabilities
    evaluator = Evaluator('chestmnist', 'val')
    metrics = evaluator.evaluate(y_score)

    # The evaluator for multi-label returns a tuple (AUC, ACC), not a dictionary.
    print(f'End of Epoch {epoch+1} - Validation AUC: {metrics[0]:.4f}, ACC: {metrics[1]:.4f}')

### Part B (Project): Training and Evaluating the `PathMNIST` model

Your turn! Follow the same steps to train the `PathMNIST` model.

#### Your Turn: Train the PathMNIST Model

You will need to:
1.  Reload the `PathMNIST` datasets using the `path_train_transform` and `path_val_test_transform` you created.
2.  Create `DataLoader`s for them.
3.  Instantiate your `path_model` and move it to the `device`.
4.  Define the `criterion`. **Hint:** The correct loss function for multi-class classification is `nn.CrossEntropyLoss()`.
5.  Define the `optimizer`.
6.  Adapt the training loop. **Hint:** `CrossEntropyLoss` expects the labels to be of a different shape. You'll need to use `.squeeze().long()` on the labels before passing them to the loss function.
7.  Adapt the validation loop. For multi-class, you can calculate accuracy directly. The predicted class is the one with the highest score, which you can find using `torch.argmax(outputs, dim=1)`.

In [ ]:
# 1. Reload datasets
path_train_dataset = # --- YOUR CODE HERE ---
path_val_dataset = # --- YOUR CODE HERE ---
path_test_dataset = # --- YOUR CODE HERE ---

# 2. Create DataLoaders
path_train_loader = # --- YOUR CODE HERE ---
path_val_loader = # --- YOUR CODE HERE ---
path_test_loader = # --- YOUR CODE HERE ---

# 3. Instantiate model
path_model = # --- YOUR CODE HERE ---

# 4. Define criterion
criterion_path = # --- YOUR CODE HERE ---

# 5. Define optimizer
optimizer_path = # --- YOUR CODE HERE ---

NUM_EPOCHS = 5

# 6 & 7. Write the training and validation loop
# --- YOUR CODE HERE ---


#### Analyzing the Results: Confusion Matrix

Accuracy is a good start, but it doesn't tell the whole story. What if the model is really good at identifying one class, but terrible at another? A **Confusion Matrix** helps us see this.

Here's how to read it:
- **Rows** represent the **Actual Class**.
- **Columns** represent the **Predicted Class**.
- The numbers on the **diagonal** (from top-left to bottom-right) are the **correct** predictions. These are our **True Positives** and **True Negatives**.
- Any number **off the diagonal** is an **error**. For example, if the row is `TUM` (Tumor) and the column is `NORM` (Normal), that number tells you how many times the model incorrectly called a real tumor a normal tissue. This is a **False Negative** and is a very dangerous error in medicine!
- If the row is `NORM` and the column is `TUM`, that's a **False Positive** (a false alarm).

A perfect model would have a bright diagonal and zeros everywhere else.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Get predictions on the test set
path_model.eval()
y_true_test = torch.tensor([]).to(device)
y_pred_test = torch.tensor([]).to(device)
with torch.no_grad():
    for images, labels in path_test_loader:
        images, labels = images.to(device), labels.squeeze().long().to(device)
        outputs = path_model(images)
        preds = torch.argmax(outputs, dim=1)
        y_true_test = torch.cat((y_true_test, labels), 0)
        y_pred_test = torch.cat((y_pred_test, preds), 0)

y_true_test = y_true_test.cpu().numpy()
y_pred_test = y_pred_test.cpu().numpy()

# Plot the confusion matrix
cm = confusion_matrix(y_true_test, y_pred_test)
class_names = list(path_info['label'].values())
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)

fig, ax = plt.subplots(figsize=(10, 10))
disp.plot(ax=ax, xticks_rotation='vertical')
plt.title('PathMNIST Confusion Matrix')
plt.show()

---
## Module 6: Advanced Concepts & Next Steps

Our simple CNN works, but we can do much better. Let's explore two advanced topics: **Transfer Learning** and **Model Interpretability**.

### Concept 1: Transfer Learning

Imagine trying to learn astrophysics without first learning to read. It would be impossible! **Transfer Learning** is a similar idea for AI. Instead of training a model from scratch (teaching it to see basic edges and shapes), we can take a powerful, state-of-the-art model that has already been trained on a massive dataset like ImageNet (which has millions of everyday images). This model already knows how to "see". We then just need to fine-tune it for our specific medical task.

We will use a **ResNet-18** model, a famous and powerful CNN architecture.

#### Your Turn: Adapt a Pre-trained ResNet-18 for PathMNIST

1.  Load a pre-trained ResNet-18 from `torchvision.models`.
2.  The pre-trained model was trained to classify 1000 things (like cats, dogs, cars). We only need to classify 9 tissue types. So, we must **replace the final fully connected layer** (`fc`) with a new one that has 9 outputs.
3.  Train this new, powerful model on `PathMNIST` and see if the performance improves.

In [ ]:
import torchvision.models as models

# Task 1: Load the pre-trained ResNet-18 model.
# The 'weights' argument tells it to download the pre-trained knowledge from ImageNet.
resnet_model = # --- YOUR CODE HERE ---

# Task 2: Replace the final layer.
# First, get the number of input features for the current final layer.
num_ftrs = resnet_model.fc.in_features
# Now, create a new Linear layer with the correct number of output classes (9).
resnet_model.fc = # --- YOUR CODE HERE ---

resnet_model = resnet_model.to(device)
print("ResNet-18 model adapted for PathMNIST.")

# Task 3: Train the new model.
# You can copy your previous training loop and just change the model variable to 'resnet_model'.
# You might find that it trains much faster and gets a higher accuracy!
# --- YOUR TRAINING LOOP CODE HERE ---

### Concept 2: Model Interpretability (XAI)

In medicine, a correct prediction isn't enough; we need to know *why* the model made that prediction. This field is called e**X**plainable **AI** (XAI). A simple way to do this is to create a **saliency map**, which is like a heat-map that shows which pixels in the input image were most important for the model's decision.

We will visualize the saliency map for a few test images using our trained ResNet model. This code is a bit more advanced, so we've provided it for you. Try to read through it and understand the main steps.

In [ ]:
def plot_saliency_map(model, image_tensor, original_image, label, pred, class_names):
    model.eval()
    # We need to tell PyTorch to calculate gradients with respect to the input image
    image_tensor.requires_grad_()

    # Get the model's scores
    scores = model(image_tensor)
    # Get the score for the class that the model predicted
    score_max_index = scores.argmax()
    score_max = scores[0, score_max_index]

    # This is the key step: calculate the gradients of the max score with respect to the image pixels
    score_max.backward()

    # The pixels with the highest gradients are the most important ones.
    saliency, _ = torch.max(image_tensor.grad.data.abs(), dim=1)

    # Plot the original image and the saliency map
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
    ax1.imshow(original_image)
    ax1.set_title(f'True: {class_names[label]}\nPred: {class_names[pred.item()]}')
    ax1.axis('off')

    ax2.imshow(saliency.cpu().squeeze(), cmap='hot')
    ax2.set_title('Saliency Map')
    ax2.axis('off')
    plt.show()

# Get a few images from the test set to visualize
unnormalized_dataset = medmnist.PathMNIST(split='test', download=True) # For getting the clean original images

for i in range(3):
    # Get the normalized image that the model expects
    img_tensor, lbl = path_test_dataset[i]
    img_tensor = img_tensor.unsqueeze(0).to(device)

    # Get the original, un-normalized image for plotting
    original_img, _ = unnormalized_dataset[i]

    # Get the ResNet model's prediction
    prediction = resnet_model(img_tensor).argmax()

    # Plot everything
    plot_saliency_map(resnet_model, img_tensor, original_img, lbl.item(), prediction, list(path_info['label'].values()))

### Workshop Conclusion

**Congratulations!** You have completed the entire workshop. You have journeyed from the basics of loading medical data to building, training, evaluating, and even interpreting advanced deep learning models. This is a huge accomplishment!

**Let's recap the key skills you've mastered:**

- **Data Handling:** You can confidently load and inspect complex datasets, understanding the critical differences between multi-class and multi-label problems.
- **Data Visualization:** You are able to display images and their labels, a fundamental skill for understanding any dataset you encounter in the future.
- **Data Preprocessing:** You wrote code to calculate dataset statistics and build robust data augmentation pipelines to prepare images for training, a process used in all state-of-the-art AI projects.
- **Model Building:** You built a Convolutional Neural Network from scratch in PyTorch, defining its architecture layer by layer.
- **Model Training:** You implemented a complete training and validation loop, teaching an AI model to recognize patterns in images.
- **Model Evaluation:** You moved beyond simple accuracy and learned to interpret professional metrics like the AUC and the Confusion Matrix to understand your model's strengths and weaknesses.
- **Advanced Techniques:** You successfully implemented Transfer Learning to leverage the power of a pre-trained model, and you peeked inside the "black box" of a neural network with saliency maps to understand what it was "seeing".

The skills you've learned over these two days are the foundation of modern artificial intelligence in medicine. We hope this workshop has sparked your curiosity and that you continue to explore this exciting field.